In [1]:
from ecostyles import EcoStyles
# Create styles instance
styles = EcoStyles()

import altair as alt
import pandas as pd
from IPython.display import display

In [2]:
# Register and enable a theme
styles.register_and_enable_theme(theme_name="article")

In [3]:
# Load crime data
crime_df = pd.read_excel('intentional_homicide_rates_regional_comparison.xlsx')

# Melt to long format
crime_df = crime_df.melt(
    id_vars='year',
    var_name='country',
    value_name='rate'
)

# Drop missing values
crime_df = crime_df.dropna(subset=['rate'])

# Clean up country names
country_mapping = {
    'el_salvador_per_100k': 'El Salvador',
    'honduras_per_100k': 'Honduras',
    'guatemala_per_100k': 'Guatemala',
    'nicaragua_per_100k': 'Nicaragua'
}
crime_df['country'] = crime_df['country'].replace(country_mapping)

crime_df

,year,country,rate
4,1994,El Salvador,134.281183
5,1995,El Salvador,138.438378
6,1996,El Salvador,116.996820
7,1997,El Salvador,112.463209
8,1998,El Salvador,94.953374
...,...,...,...
131,2016,Nicaragua,7.268158
132,2017,Nicaragua,6.823689
133,2018,Nicaragua,10.656117
134,2019,Nicaragua,7.943048


In [4]:
# Plot intentional homicide rate line chart
homicide_line_chart = alt.Chart(crime_df).mark_line().encode(
    x=alt.X('year:O',
            title='',
            axis=alt.Axis(
                values=list(range(1990, 2024, 10)),
                grid=False)),
    y=alt.Y('rate:Q',
            title='Intentional homicides per 100,000 people',
            scale=alt.Scale(domain=[0,150]),
            axis=alt.Axis(grid=False)),
    color=alt.Color('country:N', legend=None),
    tooltip=[
        alt.Tooltip('country:N', title='Country'),
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('rate:Q', title='Rate per 100k', format='.1f')
    ]
)

# End-of-line labels
last_points = crime_df.loc[crime_df.groupby('country')['year'].idxmax()]
labels = alt.Chart(last_points).mark_text(
    align='left',
    dx=12
).encode(
    x=alt.X('year:O'),
    y=alt.Y('rate:Q', scale=alt.Scale(domain=[0, 150])),
    text='country:N',
    color=alt.Color('country:N', legend=None)
)

# Bukele marker
bukele_marker = alt.Chart(pd.DataFrame({'year': [2019]})).mark_rule(
    strokeDash=[4, 3],
    strokeWidth=1.5,
    color='#676A86'
).encode(
    x='year:O'
)
bukele_label = alt.Chart(pd.DataFrame({'year': [2019], 'label': ['Bukele takes office']})).mark_text(
    align='left',
    dx=5,
    dy=-140,
    fontSize=11,
    color='#676A86'
).encode(
    x='year:O',
    text='label:N'
)

homicide_line_chart = homicide_line_chart + labels + bukele_label + bukele_marker

homicide_line_chart = styles.add_source(
    homicide_line_chart,
    'Source: World Development Indicators; UNODC; Policia Nacional Civil'
)

display({'application/vnd.vegalite.v5+json': homicide_line_chart.to_dict()}, raw=True)

In [5]:
# Save chart
styles.save(homicide_line_chart, 'visualisation', 'homicide_line_chart', width=450, height=360)